## Import

In [1]:
import torch 
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
bc = load_breast_cancer()
X,y = bc.data , bc.target
print(X.shape,y.shape)

(569, 30) (569,)


## Data spiltting

In [19]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=7)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(455, 30)
(114, 30)
(455,)
(114,)


## StandardScaler

In [20]:
Sc = StandardScaler()
X_train_sc = Sc.fit_transform(X_train)
X_test_sc = Sc.transform(X_test)

## Convert numpy to pytorch

In [ ]:
X_train_torch_sc = torch.from_numpy(X_train_sc.astype('float32'))
X_test_torch_sc = torch.from_numpy(X_test_sc.astype('float32'))
y_train_torch_sc = torch.from_numpy(y_train.astype('float32')).view(-1,1)
y_test_torch_sc = torch.from_numpy(y_test.astype('float32')).view(-1,1)


#X_train_torch_sc = torch.tensor(X_train_sc, dtype=torch.float32) not dependency 

print(X_train_torch_sc.size())
print(X_test_torch_sc.size())
print(y_train_torch_sc.size())
print(y_test_torch_sc.size())

torch.Size([455, 30])
torch.Size([114, 30])
torch.Size([455, 1])
torch.Size([114, 1])


## Model

In [77]:
class Logistic(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.linear = nn.Linear(input_size,1)

    def forward(self,X):
        y_predicted = torch.sigmoid(self.linear(X))
        return y_predicted

In [78]:
model = Logistic(X_train_torch_sc.size()[1])
learning_rate = 0.1
criterion = nn.BCELoss()
optim = torch.optim.SGD(model.parameters(),lr=learning_rate)

In [79]:
n_epoch = 1
for epoch in range(n_epoch):
    #forward
    y_predicted = model(X_train_torch_sc)
    loss = criterion(y_predicted,y_train_torch_sc)

    #backward
    loss.backward()

    #update weight
    optim.step()
    optim.zero_grad()

    print(f'epoch {epoch+1} | BCE {loss.item():.4}')

epoch 1 | BCE 0.7747


In [83]:
with torch.no_grad():
    y_hat = model(X_test_torch_sc)
    # print(y_hat.round())
    acc = y_hat.round().eq(y_test_torch_sc).sum() / float(y_test_torch_sc.size()[0])
    print(acc)

tensor(0.7807)
